### This notebook helps to understand how a shape of mata-atoms looks like, and what data collects for library.
**Note:** almost all variables are located in conf.py.

At the beginning we repeat again first part of computations for substrait. But for structures like catenaty curvs we are needed in 3d simulation, so let's expand 2d to 3d

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import meep as mp
import math

import conf

In [ ]:
print(conf.cell_X)
print(conf.cell_Y)
print(conf.cell_Z)



Also we enable symmetry cause long-term computations. How experiments show 3d computational is not a way for preparing lib. Cause even after 30 minutes of calculation it was aborted.
I want to try preparing a simulation field in 3d however run in 2d. Is it real? I checked, that is not possible on my PC.

We need a function that will build a three-dimensional meta-atom, then cut it and give only the coordinates for constructing this section. Then the 4 cross-section points will be transferred to 2d calculation. Presumably, 1 shape of a meta-atom can describe 2-3 cross-sections.

In [ ]:
pml_layers = [mp.PML(thickness=conf.in_pml, direction=mp.X)]
cell = mp.Vector3(conf.cell_X, conf.cell_Y, 0)
symmetries = [mp.Mirror(mp.Y)]
geometry = [mp.Block(
    material=conf.in_material,
    size=mp.Vector3(conf.in_sub, mp.inf, conf.cell_Z),
    center=conf.sub_center,
)]
sources = [mp.Source(
    mp.ContinuousSource(conf.inc_fcen, fwidth=conf.inc_df),
    component=mp.Ez,
    center=conf.source_center,
    size=mp.Vector3(x=0, y=conf.cell_Y, z=conf.cell_Z),
)]

sim = mp.Simulation(
    cell_size=cell,
    boundary_layers=pml_layers,
    geometry=geometry,
    sources=sources,
    resolution=conf.resolution,
    symmetries = symmetries,
)
flux_obj = sim.add_flux(
            conf.inc_fcen, 0, 1, mp.FluxRegion(center=conf.flux_center, size=mp.Vector3(y=conf.cell_Y, z=conf.cell_Z))
)
%matplotlib inline
plt.figure(dpi=100)
sim.plot2D(
    output_plane = mp.Volume(center=mp.Vector3(0, 0, 0), size=mp.Vector3(10,10,0)) 
)
plt.show()

## Note
By changing the values for **size** in the output_plane parameter, we can plot graphs in various directions and check whether our geometry is constructed correctly.

In [ ]:
sim.run(until=conf.steps) # run 1st part of simulation
input_flux = mp.get_fluxes(flux_obj) # saving flux from sub in input_flux
sim.reset_meep() # purge all adjustments of simulation

The following code allows us to define points for a curved curve.
This approach allows you to pre-configure the geometry description and verify its correctness by building a plot.

In [ ]:
# curve line parameters
a = 1.5  # tang coefficient 
x_max = 5  # range x
x_min = -x_max
width = 1  # width between top and bottom curve 
num_points = 50  # resolution 

# Generating data for the upper and lower curves
x = np.linspace(x_min, x_max, num_points)
y_top = a * np.cosh(x / a)  # the upper curve
y_bottom = y_top - width # the bottom curve

# Coordinates of the ends for the closure
y_top_left = a * np.cosh(x_min / a)
y_top_right = a * np.cosh(x_max / a)
y_bottom_left = y_top_left - width # It is only necessary for plotting, these coordinates are not transmitted to the simulation.
y_bottom_right = y_top_right - width # It is only necessary for plotting, these coordinates are not transmitted to the simulation.

# Building a closed form
plt.figure(figsize=(8, 6))
plt.plot(x, y_top, label='Upper Catenary Curve', color='red')
plt.plot(x, y_bottom, label='Lower Catenary Curve', color='red')
plt.plot([x_min, x_min], [y_bottom_left, y_top_left], color='red', label='Left Edge')
plt.plot([x_max, x_max], [y_bottom_right, y_top_right], color='red', label='Right Edge')

# Settings
plt.title('Closed Shape Bounded by Two Catenary Curves')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

def __init__(self,
             vertices,
             height,
             axis=Vector3<0.0, 0.0, 1.0>,
             center=None,
             sidewall_angle=0,
             **kwargs):
Construct a Prism.

vertices [list of Vector3] — The vertices that make up the prism. They must lie in a plane that's perpendicular to the axis. Note that infinite lengths are not supported. To simulate infinite geometry, just extend the edge of the prism beyond the cell.

height [number] — The prism thickness, extruded in the direction of axis. mp.inf can be used for infinite height. No default value.

axis [Vector3] — The axis perpendicular to the prism. Defaults to Vector3(0,0,1).

center [Vector3] — If center is not specified, then the coordinates of the vertices define the bottom of the prism with the top of the prism being at the same coordinates shifted by height*axis. If center is specified, then center is the coordinates of the centroid of all the vertices (top and bottom) of the resulting 3d prism so that the coordinates of the vertices are shifted accordingly.

sidewall_angle [number] — The sidewall angle of the prism in units of radians. Default is 0.

Great, the appearance of the cell has been checked. But for the calculation, we only need sections of this geometry. And in global coordinate systems, the cross section of this geometry degenerates. 
rectangles (or rectangles) that are offset along the Z axis.

X for an atom is always equal to its height (x=const), only the positions of the vertices change. Let's write a function that will, according to the geometry description, output these vertices in accordance with the number of sections.

In [ ]:
number_sections = 10
#x_begin = solve_for_max_width(width,a)
x_begin = 0
x_sections = np.linspace(x_begin, x_max, number_sections, endpoint=True, dtype=float) # points of the section plane (symmetrical with respect to 0)
width_wide = [] # Here we collect the widths of the sections
x_centers = [] # Here we collect their relative centers.

def solve_for_x(x0, a, width):
    cosh_diff = math.cosh(x0 / a) - width / a
    if cosh_diff < 1:
        raise ValueError("Invalid value: the expression under arcosh must be >= 1")
    
    # Calculating x
    return a * math.acosh(cosh_diff)

def solve_for_max_width(width, a):
     # Проверяем, что значение внутри arcosh корректное (должно быть >= 1)
     if (width) < 0: # width must be > 0 
        raise ValueError("Invalid value: width must be > 0.")
     # Calculating x
     x = a * math.acosh(1 + width / a)
     return x     

In [ ]:
print(f'width: {width}, a: {a}')
test = solve_for_max_width(width, a)
print(f'test: {test}')

In [ ]:
x_sections

In [ ]:
def not_split(x_sections, a ,width):
    return (a * np.cosh(x_sections / a) - width) <= a

In [ ]:
for i in range(number_sections):
    result = not_split(x_sections[i], a, width)
    print(result)

In [ ]:
for i in range(number_sections):    
    if not_split(x_sections[i], a, width):
        x_centers.append(0)
        width_wide.append(x_sections[i]*2)     
    else:
        find_width = x_sections[i] - solve_for_x(x_sections[i], a, width )
        width_wide.append(find_width) 
        x_centers.append(x_sections[i] - find_width/2)

In [ ]:
print(f'a coefficient equals: {a} and width equals {width}')
for i in range(number_sections):
    print(f'x_sections {i} is: {x_sections[i]:.2f} | x_centers {i} is: {x_centers[i]:.2f} | width_wide for this center is: {width_wide[i]:.2f} ')

We need to count each section and then clean the assembly. The height of the section does not change, but the location of the centers and the width of the sections change.

In [ ]:
# We will store the results in a worksheet. First, initialize it.
mode_tran = np.empty(number_sections)
mode_phase = np.empty(number_sections)

geometry.append(mp.Block(
    material=conf.at_material,
    size=mp.Vector3(0,0,0),
    center=mp.Vector3(0,0,0),
))

geometry.append(mp.Block(
    material=conf.at_material,
    size=mp.Vector3(0,0,0),
    center=mp.Vector3(0,0,0),
))

### Implementation of the main code
We go through all the sections and calculate the phase shift and transmission in them.

In [ ]:
for i in range(number_sections):
    geometry[1].size = mp.Vector3(x=conf.at_hei, y = width_wide[i], z = mp.inf)
    geometry[2].size = mp.Vector3(x=conf.at_hei, y = width_wide[i], z = mp.inf)
    if x_centers[i] == 0:   
        geometry[1].center = mp.Vector3(x = conf.at_center[0], y = 0, z = 0)
        geometry[2].center = mp.Vector3(x = conf.at_center[0], y = 0, z = 0)
    else:
        geometry[1].center = mp.Vector3(x = conf.at_center[0], y = x_centers[i], z = 0)
        geometry[2].center = mp.Vector3(x = conf.at_center[0], y = -x_centers[i], z = 0)
        
    sim = mp.Simulation(
        cell_size=cell,
        boundary_layers=pml_layers,
        geometry=geometry,
        sources=sources,
        resolution=conf.resolution,
    )
    flux_obj = sim.add_flux(
        conf.inc_fcen, 0, 1, mp.FluxRegion(
            center=mp.Vector3(x=conf.flux_center[0]), 
            size=mp.Vector3(y=conf.cell_Y)
        )
    )
    sim.run(until=conf.steps)
    
    res = sim.get_eigenmode_coefficients(
        flux_obj, [1], eig_parity=mp.ODD_Z + mp.EVEN_Y
    )
    coeffs = res.alpha
    
    mode_tran[i] = abs(coeffs[0, 0, 0]) ** 2 / input_flux[0]
    mode_phase[i] = np.angle(coeffs[0, 0, 0])
    if mode_phase[i] > 0:
        mode_phase[i] -= 2 * np.pi
        
    # Смотрим на расчет каждый шаг, заодно проверяем смещение сечения. 
    %matplotlib inline
    plt.figure(dpi=100)
    sim.plot2D(fields=mp.Ez)
    plt.show()
    sim.reset_meep() # purge all adjustments of simulation
       

In [ ]:
print(mode_tran)
print(mode_phase)

In [ ]:
width_wide

In [ ]:
# Plot for visualise width_wide and mode_phase
plt.figure(figsize=(8, 6))  
plt.plot(width_wide, mode_phase, marker='o', linestyle='-', color='b', label='A vs mode_phase')

# Settings
plt.title('Зависимость A от mode_phase', fontsize=14)
plt.xlabel('x_section', fontsize=12)
plt.ylabel('mode_phase', fontsize=12)
plt.grid(True) 
plt.legend() 
plt.tight_layout()  # Оптимизируем расположение элементов

# Показываем график
plt.show()

In [ ]:
mode_phase.size

In the next part, we need to calculate a large number of catenary curves by changing the width and a parameters, and somehow summarizing the calculations in 2d sections.